# Day 6: SQLAlchemy, Relationships and API Docs
**ORM basics, relationships, transactions and Swagger / OpenAPI docs**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.
* Sections marked **NEW (v4)** were added in program v4.

## 1. Setup

In [ ]:
# ===== Option A: Google Colab (active) =====
!pip install -q fastapi httpx sqlalchemy pydantic

# ===== Option B: VS Code / local (run in your activated venv terminal) =====
# pip install fastapi uvicorn httpx sqlalchemy pydantic
import sqlalchemy
print("SQLAlchemy", sqlalchemy.__version__)

## 2. ORM basics: one table
SQLite is a database stored in a single file. No server needed, so it works in Colab and VS Code alike.

In [ ]:
import os
from sqlalchemy import create_engine, select, String
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, Session

if os.path.exists("intro.db"):
    os.remove("intro.db")
engine0 = create_engine("sqlite:///intro.db", echo=False)   # echo=True prints the SQL

class Base0(DeclarativeBase):
    pass

class BookRow(Base0):
    __tablename__ = "books"
    id: Mapped[int] = mapped_column(primary_key=True)
    title: Mapped[str] = mapped_column(String(200))
    author: Mapped[str]
    pages: Mapped[int]

Base0.metadata.create_all(engine0)
with Session(engine0) as s:
    s.add_all([BookRow(title="Ikigai", author="Garcia", pages=208),
               BookRow(title="Deep Work", author="Newport", pages=296),
               BookRow(title="Atomic Habits", author="Clear", pages=320),
               BookRow(title="The Little Prince", author="Saint-Exupery", pages=96)])
    s.commit()

with Session(engine0) as s:
    stmt = select(BookRow).where(BookRow.pages > 200).order_by(BookRow.title)
    print([b.title for b in s.scalars(stmt)])
    book = s.get(BookRow, 1)
    book.pages = 210                  # update: change the attribute
    s.delete(s.get(BookRow, 4))       # delete
    s.commit()
    print([(b.title, b.pages) for b in s.scalars(select(BookRow))])

### TODO 1: Turn on `echo=True` in `create_engine` and rerun a query. Read the SQL that SQLAlchemy writes for you.

In [ ]:
# TODO: write your code here

## 3. Models with relationships

In [ ]:
%%writefile models.py
from sqlalchemy import Column, ForeignKey, String, Table
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship

class Base(DeclarativeBase):
    pass

book_tags = Table(
    "book_tags", Base.metadata,
    Column("book_id", ForeignKey("books.id"), primary_key=True),
    Column("tag_id", ForeignKey("tags.id"), primary_key=True),
)

class Author(Base):
    __tablename__ = "authors"
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100), unique=True)
    books: Mapped[list["Book"]] = relationship(back_populates="author", cascade="all, delete-orphan")

class Book(Base):
    __tablename__ = "books"
    id: Mapped[int] = mapped_column(primary_key=True)
    title: Mapped[str] = mapped_column(String(200))
    pages: Mapped[int] = mapped_column(default=0)
    author_id: Mapped[int] = mapped_column(ForeignKey("authors.id"))
    author: Mapped[Author] = relationship(back_populates="books")
    tags: Mapped[list["Tag"]] = relationship(secondary=book_tags, back_populates="books")

class Tag(Base):
    __tablename__ = "tags"
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(50), unique=True)
    books: Mapped[list[Book]] = relationship(secondary=book_tags, back_populates="tags")

class Account(Base):
    __tablename__ = "accounts"
    id: Mapped[int] = mapped_column(primary_key=True)
    owner: Mapped[str]
    balance: Mapped[int]

In [ ]:
import importlib, models
importlib.reload(models)
from models import Base, Author, Book, Tag, Account
from sqlalchemy.orm import selectinload

if os.path.exists("library.db"):
    os.remove("library.db")
engine = create_engine("sqlite:///library.db")
Base.metadata.create_all(engine)
print(list(Base.metadata.tables))

## 4. Create linked data

In [ ]:
with Session(engine) as s:
    fiction, travel, tech = Tag(name="fiction"), Tag(name="travel"), Tag(name="tech")
    ito = Author(name="Ito")
    rao = Author(name="Rao")
    ito.books = [Book(title="Tokyo Nights", pages=240, tags=[fiction, travel]),
                 Book(title="Rain in Kyoto", pages=180, tags=[fiction])]
    rao.books = [Book(title="Monsoon Code", pages=320, tags=[tech]),
                 Book(title="Chai Stories", pages=150, tags=[fiction, travel])]
    s.add_all([ito, rao])
    s.commit()

with Session(engine) as s:
    stmt = select(Author).options(selectinload(Author.books).selectinload(Book.tags))
    for a in s.scalars(stmt):
        for b in a.books:
            print(f"{a.name:<4} | {b.title:<14} | {[t.name for t in b.tags]}")

## 5. Query across relationships

In [ ]:
with Session(engine) as s:
    stmt = select(Book).join(Book.author).where(Author.name == "Rao")
    print("By Rao:", [b.title for b in s.scalars(stmt)])

    stmt = select(Book).join(Book.tags).where(Tag.name == "fiction").order_by(Book.title)
    print("Fiction:", [b.title for b in s.scalars(stmt)])

## 6. Constraints and IntegrityError

In [ ]:
from sqlalchemy.exc import IntegrityError
with Session(engine) as s:
    try:
        s.add(Author(name="Ito"))        # name must be unique
        s.commit()
    except IntegrityError as e:
        s.rollback()
        print("Rejected:", str(e.orig))

## 7. Transactions: a safe transfer

In [ ]:
with Session(engine) as s:
    s.add_all([Account(owner="Hana", balance=1000), Account(owner="Arjun", balance=500)])
    s.commit()

def transfer(from_id, to_id, amount):
    with Session(engine) as s:
        try:
            a, b = s.get(Account, from_id), s.get(Account, to_id)
            if a is None or b is None:
                raise ValueError("Account not found")
            a.balance -= amount
            if a.balance < 0:
                raise ValueError("Insufficient balance")
            b.balance += amount
            s.commit()
            print(f"Transferred {amount}")
        except ValueError as e:
            s.rollback()
            print("Failed:", e)

def balances():
    with Session(engine) as s:
        return [(x.owner, x.balance) for x in s.scalars(select(Account))]

transfer(1, 2, 300); print(balances())
transfer(1, 2, 9999); print(balances())   # unchanged after rollback
transfer(1, 7, 10); print(balances())

### TODO 2: Add a `transfer_log` list that records successful transfers only.

In [ ]:
# TODO: write your code here

## 8. A documented FastAPI app on top

In [ ]:
%%writefile library_api.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from sqlalchemy import create_engine, select
from sqlalchemy.orm import Session, selectinload
from models import Author, Book

engine = create_engine("sqlite:///library.db")

tags_metadata = [
    {"name": "Books", "description": "Browse books with their author and tags."},
    {"name": "Authors", "description": "Authors and the books they wrote."},
]
app = FastAPI(title="Library API", version="1.2.0",
              description="A small library service built in the Day 6 class.",
              openapi_tags=tags_metadata)

class BookShort(BaseModel):
    model_config = {"from_attributes": True}
    id: int
    title: str = Field(examples=["Tokyo Nights"])

class BookOut(BookShort):
    pages: int
    author_name: str
    tags: list[str]

class AuthorOut(BaseModel):
    model_config = {"from_attributes": True}
    id: int
    name: str
    books: list[BookShort]

@app.get("/books", tags=["Books"], summary="List all books", response_model=list[BookOut])
def list_books():
    """Returns every book with its author name and tag names."""
    with Session(engine) as s:
        rows = s.scalars(select(Book).options(selectinload(Book.author), selectinload(Book.tags)))
        return [BookOut(id=b.id, title=b.title, pages=b.pages, author_name=b.author.name,
                        tags=[t.name for t in b.tags]) for b in rows]

@app.get("/authors/{author_id}", tags=["Authors"], summary="Get one author with books",
         response_model=AuthorOut, responses={404: {"description": "Author not found"}})
def get_author(author_id: int):
    """Returns one author and the short list of their books."""
    with Session(engine) as s:
        a = s.get(Author, author_id, options=[selectinload(Author.books)])
        if a is None:
            raise HTTPException(404, "Author not found")
        return AuthorOut.model_validate(a)

In [ ]:
# ===== Option A: Google Colab (active) =====
import library_api
importlib.reload(library_api)
from fastapi.testclient import TestClient
client = TestClient(library_api.app)

# ===== Option B: VS Code =====
# Terminal: uvicorn library_api:app --reload
# Then open http://127.0.0.1:8000/docs and http://127.0.0.1:8000/redoc

print(client.get("/books").json()[0])
print(client.get("/authors/1").json())
print(client.get("/authors/99").status_code)

## 9. Look inside the OpenAPI contract

In [ ]:
import json
spec = library_api.app.openapi()
print(spec["info"])
print("Paths:", list(spec["paths"]))
print("Schemas:", list(spec["components"]["schemas"]))
print("404 documented:", "404" in spec["paths"]["/authors/{author_id}"]["get"]["responses"])
with open("openapi.json", "w") as f:
    json.dump(spec, f, indent=2)
print("Saved openapi.json")

### TODO 3: Add a `GET /tags/{name}/books` endpoint with a summary, a tag and a 404 response, then check it appears in the spec.

In [ ]:
# TODO: write your code here

### TODO 4 (SmartDesk step 5): model `Customer` (one-to-many) `Ticket` (one-to-many) `Comment` in SQLite, and close a ticket plus add its closing comment in one transaction.

In [ ]:
# TODO: write your code here

---
**Ganbatte (頑張って): *Do your best***

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)